# Artifact Medical AI - Skin Lesion Training\n\nColab에서 피부 병변 이미지 분류 모델을 빠르게 학습/검증하고, `fastapi`에 넣을 모델 파일을 생성하는 템플릿입니다.\n\n기본 가정:\n- Google Drive에 데이터셋이 있음\n- `metadata.csv`에 이미지 ID와 라벨 컬럼이 있음\n- 이미지 파일은 `images/` 폴더 안에 있음\n- 라벨 값은 백엔드 `disease.disease_code`와 같은 `nv`, `mel`, `bkl`, `bcc`, `akiec`, `df`, `vasc` 형식

In [ ]:
# 1. Google Drive 연결\nfrom google.colab import drive\ndrive.mount('/content/drive')

In [ ]:
# 2. 기본 설정\n# 아래 경로는 본인 Drive 데이터 위치에 맞게 수정하세요.\nDATA_ROOT = '/content/drive/MyDrive/artifact-medical-ai/ham10000'\nMETADATA_CSV = f'{DATA_ROOT}/metadata.csv'\nIMAGE_DIR = f'{DATA_ROOT}/images'\nOUTPUT_DIR = '/content/drive/MyDrive/artifact-medical-ai/model-output'\n\n# HAM10000 metadata 기준 기본 컬럼명입니다.\n# 다른 데이터셋이면 여기를 바꾸면 됩니다.\nIMAGE_ID_COL = 'image_id'\nLABEL_COL = 'dx'\n\nBATCH_SIZE = 32\nIMAGE_SIZE = 224\nEPOCHS = 3\nLEARNING_RATE = 1e-4\nSEED = 42

In [ ]:
# 3. 라이브러리 로드\nimport json\nimport os\nimport random\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nimport torch.nn as nn\nfrom PIL import Image\nfrom sklearn.model_selection import train_test_split\nfrom torch.utils.data import Dataset, DataLoader\nfrom torchvision import models, transforms\n\nrandom.seed(SEED)\nnp.random.seed(SEED)\ntorch.manual_seed(SEED)\ndevice = torch.device('cuda' if torch.cuda.is_available() else 'cpu')\ndevice

In [ ]:
# 4. 메타데이터 확인\ndf = pd.read_csv(METADATA_CSV)\nprint(df.shape)\ndisplay(df.head())\nprint(df[LABEL_COL].value_counts())

In [ ]:
# 5. 이미지 경로 만들기\ndef resolve_image_path(image_id):\n    image_id = str(image_id)\n    candidates = [\n        Path(IMAGE_DIR) / image_id,\n        Path(IMAGE_DIR) / f'{image_id}.jpg',\n        Path(IMAGE_DIR) / f'{image_id}.jpeg',\n        Path(IMAGE_DIR) / f'{image_id}.png',\n    ]\n    for path in candidates:\n        if path.exists():\n            return str(path)\n    return None\n\ndf['image_path'] = df[IMAGE_ID_COL].apply(resolve_image_path)\ndf = df.dropna(subset=['image_path', LABEL_COL]).reset_index(drop=True)\nprint(df.shape)\ndisplay(df[[IMAGE_ID_COL, LABEL_COL, 'image_path']].head())

In [ ]:
# 6. 라벨맵 생성\n# 백엔드 disease.disease_code와 같은 코드가 들어가야 합니다.\nlabel_names = ['nv', 'mel', 'bkl', 'bcc', 'akiec', 'df', 'vasc']\nlabel_names = [label for label in label_names if label in set(df[LABEL_COL])]\nlabel_to_idx = {label: idx for idx, label in enumerate(label_names)}\nidx_to_label = {idx: label for label, idx in label_to_idx.items()}\n\ndf = df[df[LABEL_COL].isin(label_to_idx)].copy()\ndf['label_idx'] = df[LABEL_COL].map(label_to_idx)\n\nprint(label_to_idx)\nprint(df[LABEL_COL].value_counts())

In [ ]:
# 7. train/valid 분리\ntrain_df, valid_df = train_test_split(\n    df,\n    test_size=0.2,\n    random_state=SEED,\n    stratify=df['label_idx']\n)\nprint(train_df.shape, valid_df.shape)

In [ ]:
# 8. Dataset/DataLoader\ntrain_transform = transforms.Compose([\n    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),\n    transforms.RandomHorizontalFlip(),\n    transforms.RandomRotation(10),\n    transforms.ToTensor(),\n    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),\n])\n\nvalid_transform = transforms.Compose([\n    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),\n    transforms.ToTensor(),\n    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),\n])\n\nclass SkinLesionDataset(Dataset):\n    def __init__(self, frame, transform):\n        self.frame = frame.reset_index(drop=True)\n        self.transform = transform\n\n    def __len__(self):\n        return len(self.frame)\n\n    def __getitem__(self, idx):\n        row = self.frame.iloc[idx]\n        image = Image.open(row['image_path']).convert('RGB')\n        image = self.transform(image)\n        label = int(row['label_idx'])\n        return image, label\n\ntrain_loader = DataLoader(\n    SkinLesionDataset(train_df, train_transform),\n    batch_size=BATCH_SIZE,\n    shuffle=True,\n    num_workers=2\n)\nvalid_loader = DataLoader(\n    SkinLesionDataset(valid_df, valid_transform),\n    batch_size=BATCH_SIZE,\n    shuffle=False,\n    num_workers=2\n)

In [ ]:
# 9. 모델 생성: timm EfficientNet-B0 전이학습
num_classes = len(label_names)
model = timm.create_model('efficientnet_b0', pretrained=True, num_classes=num_classes)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)


In [ ]:
# 10. 학습/검증 함수\ndef run_epoch(loader, training):\n    model.train(training)\n    total_loss = 0.0\n    total_correct = 0\n    total_count = 0\n\n    for images, labels in loader:\n        images = images.to(device)\n        labels = labels.to(device)\n\n        with torch.set_grad_enabled(training):\n            outputs = model(images)\n            loss = criterion(outputs, labels)\n\n            if training:\n                optimizer.zero_grad()\n                loss.backward()\n                optimizer.step()\n\n        preds = outputs.argmax(dim=1)\n        total_loss += loss.item() * labels.size(0)\n        total_correct += (preds == labels).sum().item()\n        total_count += labels.size(0)\n\n    return total_loss / total_count, total_correct / total_count\n\nbest_valid_acc = 0.0\nfor epoch in range(1, EPOCHS + 1):\n    train_loss, train_acc = run_epoch(train_loader, training=True)\n    valid_loss, valid_acc = run_epoch(valid_loader, training=False)\n    print(f'Epoch {epoch}: train_loss={train_loss:.4f}, train_acc={train_acc:.4f}, valid_loss={valid_loss:.4f}, valid_acc={valid_acc:.4f}')\n    best_valid_acc = max(best_valid_acc, valid_acc)

In [ ]:
# 11. 단일 이미지 추론 테스트\nmodel.eval()\nsample = valid_df.sample(1, random_state=SEED).iloc[0]\nimage = Image.open(sample['image_path']).convert('RGB')\nx = valid_transform(image).unsqueeze(0).to(device)\n\nwith torch.no_grad():\n    probs = torch.softmax(model(x), dim=1)[0].cpu().numpy()\n\ntop_idx = probs.argsort()[::-1][:3]\nprint('answer:', sample[LABEL_COL])\nfor idx in top_idx:\n    print(idx_to_label[int(idx)], float(probs[idx]))\ndisplay(image)

In [ ]:
# 12. 모델/라벨맵 저장
os.makedirs(OUTPUT_DIR, exist_ok=True)
model_path = f'{OUTPUT_DIR}/model.pth'
label_map_path = f'{OUTPUT_DIR}/label_map.json'
preprocess_path = f'{OUTPUT_DIR}/preprocess_config.json'

# fastapi/main.py는 timm efficientnet_b0 state_dict를 바로 로드한다.
torch.save(model.state_dict(), model_path)

with open(label_map_path, 'w', encoding='utf-8') as f:
    json.dump({str(k): v for k, v in idx_to_label.items()}, f, ensure_ascii=False, indent=2)

with open(preprocess_path, 'w', encoding='utf-8') as f:
    json.dump({
        'model_name': 'efficientnet_b0',
        'image_size': IMAGE_SIZE,
        'mean': [0.485, 0.456, 0.406],
        'std': [0.229, 0.224, 0.225]
    }, f, ensure_ascii=False, indent=2)

print(model_path)
print(label_map_path)
print(preprocess_path)
